# 59. Causal BO Benchmark：CBO・GP-BO・Random・GP-SCMを比較して読む

このNotebookはPhase 6で実装済みの`run_causal_benchmark`を**そのまま呼び出し**、因果構造既知の合成SCMで基本の3手法と任意追加のGP-SCMを比較して、実測した最良値・Simple Regret・累積コストを可視化します。ベンチマーク本体をNotebook内で再実装しません。

**このNotebookで確認すること**：共通初期介入、評価回数の公平性、介入集合が異なる3手法の違い、コスト軸での見え方、複数シードの集計。

関連：[Theory 28章](../../docs/theory/28_causal_bayesian_optimization.md) /
[Guide 28章](../../docs/guide/28_causal_bayesian_optimization.md) /
[ベンチマークの検証範囲](../../docs/benchmarks/README.md) /
[実際のAsk-and-Tell例](58_causal_bayesian_optimization.ipynb)

**実行の流れ：** 既知SCM → 同じ初期介入 → CBO / 固定集合GP-BO / Random → 介入実測 → 真の目的値・最良値・regret・コスト → 曲線集計。


## 1. 何を比較し、何を比べないか

`X → M → Y ← Z` の既知SCMで、`M=0.25+0.75X`、`Y=-(M-0.8)^2-0.5(Z-0.65)^2` を評価します。joint介入`do(X,Z)`を使えば最適値は0です。

- **CBO**は`do(X)`、`do(Z)`、`do(X,Z)`を選べます。
- **GP-BO**は`do(X,Z)`に固定して値だけ最適化します。部分介入の結果を全介入GPの学習データに混ぜません。
- **Random**は3種類の集合と介入値をランダムに選びます。

各シードの初期介入は**3集合×2件=6件で完全共通**。その後は各手法に同じ**追加試行回数**を与えます。ただし、費用総額は一致しません。異なる行動空間を許していることも明示的な比較上の制約です。


In [ ]:
import torch
import matplotlib.pyplot as plt

from robotorchan.benchmarks import (
    make_causal_synthetic_scenario,
    run_causal_benchmark,
    summarize_curves,
)

torch.manual_seed(0)
scenario = make_causal_synthetic_scenario()
print("DAG:", scenario.graph.topological_order)
print("集合別の介入費用:", {"+".join(sorted(k)): v for k, v in scenario.costs.items()})
print("既知の最適値:", scenario.optimal_value)


## 2. 同じ初期介入から3手法を実行する

ここでは2シード、集合ごと2回の初期試行、追加2回の小さな例を実行します。候補最適化は軽量な`optimizer="sobol"`にしています。実務比較では複数シード、十分な予算、対応手法の設定を増やして評価してください。


In [ ]:
trajectories = run_causal_benchmark(
    seeds=(0, 1),
    n_initial_per_set=2,
    n_iterations=2,
    scenario=scenario,
    optimizer="sobol",
    num_restarts=2,
    raw_samples=16,
)
assert len(trajectories) == 6
for seed in (0, 1):
    group = [t for t in trajectories if t.seed == seed]
    assert len(group) == 3
    assert group[0].steps[:6] == group[1].steps[:6] == group[2].steps[:6]
print("共通初期介入は各シードで一致しています。")
for item in trajectories:
    last = item.steps[-1]
    print(item.method, "seed", item.seed,
          "best", round(last.best_truth, 5),
          "regret", round(last.simple_regret, 5),
          "cost", round(last.cumulative_cost, 2))


## 3. Best value・Simple Regret・Costを読む

最大化なので、真の目的値の累積最良値`b_t`は減少しません。真の最適値が0なので、`simple_regret=max(0,-b_t)`は改善すると小さくなります。`cumulative_cost`は実験費用の和です。

初期6試行はどの手法も同じため、グラフの左側が重なることは当然です。「CBOの実測性能が優れる」といった結論をこの小例だけで述べてはいけません。


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for method in ("cbo", "gp_bo", "random"):
    item = next(t for t in trajectories if t.method == method and t.seed == 0)
    evaluations = [step.evaluation for step in item.steps]
    axes[0].plot(evaluations, item.best_curve.numpy(), marker=".", label=method)
    axes[1].plot(evaluations, item.regret_curve.numpy(), marker=".", label=method)
    axes[2].plot(evaluations, item.cost_curve.numpy(), marker=".", label=method)
axes[0].set(title="Best true Y", xlabel="Evaluation", ylabel="Best Y")
axes[1].set(title="Simple regret", xlabel="Evaluation", ylabel="Regret")
axes[2].set(title="Cumulative cost", xlabel="Evaluation", ylabel="Cost")
axes[0].legend()
plt.tight_layout()
plt.show()


## 4. 同じコストで比較すると見え方が変わる

コスト対最良値のグラフは、同じ回数だけ実験した場合の比較とは異なります。安い`do(X)`が多い手法は左側に点が集まるかもしれません。ただし、各手法が同じ**コスト上限**まで探索して止まる設計ではないため、完全な等コストベンチマークではありません。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for method in ("cbo", "gp_bo", "random"):
    item = next(t for t in trajectories if t.method == method and t.seed == 0)
    ax.step(item.cost_curve.numpy(), item.best_curve.numpy(),
            where="post", marker="o", label=method)
ax.set(xlabel="Cumulative intervention cost",
       ylabel="Best true Y", title="Same evaluation count, different cost")
ax.legend()
plt.show()


## 5. シードごとに対応づけて集計する

個別のランダムシードの曲線に偶然の変動があっても、結論を急がないことが重要です。ここでは既存の`summarize_curves`を再利用して平均・標準偏差を出します。`n=2`は集計方法の説明であり、統計的優位性を証明する規模ではありません。


In [ ]:
for method in ("cbo", "gp_bo", "random"):
    curves = {t.seed: t.best_curve for t in trajectories if t.method == method}
    summary = summarize_curves(curves)
    print(method, "seed", summary.seeds,
          "final mean best", round(summary.mean[-1].item(), 5),
          "final std", round(summary.std[-1].item(), 5))


## 6. 実際のベンチマークを実施するには

本番の比較実験はNotebookではなく既存スクリプト`benchmarks/causal_bo.py`から実行してください。

```bash
python benchmarks/causal_bo.py --seeds 0,1,2 --n-initial-per-set 2 --iterations 5
```

出力のCSVは各実験の`intervention_set`、`intervention_values`、`best_truth`、`simple_regret`、`cumulative_cost`などを記録し、集計JSONはシード間の曲線平均・標準偏差を記録します。

**結論の限界**：今回のGP-BOは全変数の固定集合だけを最適化します。CBOは集合を選べるので、両者の差には「探索可能な行動集合の差」も含まれます。未知のDAG発見、観測データからの因果効果同定、実物装置の安全性は評価していません。


## 7. GP-SCMを同じ初期介入データで比較する

新しく追加した`gp_scm`方式は、既知DAGに沿って各ノードの条件付きGPを学習し、介入した結果のサンプルからMC期待改善量を評価します。
**比較ランナーの既定の3手法は変わりません**。GP-SCMを追加する場合だけ
`methods=("gp_scm",)` を明示して実行できます。

この例ではGP-SCMだけに追加の観測用データを与えません。4手法とも**同一シードで実際に実施した初期6件のHard intervention**を受け取り、GP-SCMは記録済みの中間状態 `M` も利用して機構学習します。
集合別CBOは介入集合ごとの実測目的値を使い、固定集合GP-BOはjoint介入の結果だけを使います。したがって、**同じ行から異なる情報を学習する**比較です。

まずGP-SCMだけを追加実行し、既に作った3手法の結果と結合します。
説明用に2シード・追加2回・MCサンプル24個の小さな条件とします。通常の大規模な性能評価に十分な数ではありません。

In [ ]:
gp_scm_runs = run_causal_benchmark(
    seeds=(0, 1),
    n_initial_per_set=2,
    n_iterations=2,
    scenario=scenario,
    methods=("gp_scm",),
    optimizer="sobol",
    raw_samples=12,
    mc_samples=24,
    mc_seed=17,
)
assert len(gp_scm_runs) == 2
for run in gp_scm_runs:
    reference = next(t for t in trajectories if t.method == "cbo" and t.seed == run.seed)
    assert run.steps[:6] == reference.steps[:6]  # 完全に一致した実介入初期履歴
    assert run.initial_points == 6 and len(run.steps) == 8
print("GP-SCMと従来3方式の初期実介入データは一致しています。")

all_runs = (*trajectories, *gp_scm_runs)
methods_4 = ("cbo", "gp_bo", "random", "gp_scm")
for method in methods_4:
    row = next(t for t in all_runs if t.method == method and t.seed == 0)
    last = row.steps[-1]
    print(f"{method:7s}: best={last.best_truth:.5f}, "
          f"regret={last.simple_regret:.5f}, cost={last.cumulative_cost:.2f}")

## 8. 4方式の結果を実験回数・コストの両軸で読む

左図は**実験回数に対する最良の実測目的値**です。中央図は同じ回数のSimple Regret、
右図は**累積費用に対する最良値**です。費用が一定ではないため、3図の解釈は異なります。

同じ6回の初期試験では曲線が重なります。追加試験で差が出ても、その理由には
使用する介入集合の違い、利用する測定項目の違い、確率モデルの違いが混ざります。
「因果構造を使った効果」だけを単独で識別した結果ではありません。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for method in methods_4:
    run = next(t for t in all_runs if t.method == method and t.seed == 0)
    count = [step.evaluation for step in run.steps]
    axes[0].plot(count, run.best_curve.numpy(), marker="o", label=method)
    axes[1].plot(count, run.regret_curve.numpy(), marker="o", label=method)
    axes[2].step(run.cost_curve.numpy(), run.best_curve.numpy(),
                 where="post", marker="o", label=method)

axes[0].set(title="Best observed outcome", xlabel="Evaluation count", ylabel="Best Y")
axes[1].set(title="Simple regret (lower is better)", xlabel="Evaluation count",
            ylabel="Regret")
axes[2].set(title="Best Y vs cumulative cost", xlabel="Cumulative intervention cost",
            ylabel="Best Y")
axes[0].legend()
axes[2].legend()
plt.tight_layout()
plt.show()

for method in methods_4:
    curves = {t.seed: t.regret_curve for t in all_runs if t.method == method}
    report = summarize_curves(curves)
    print(method, "final mean regret", round(report.mean[-1].item(), 5),
          "std", round(report.std[-1].item(), 5))

## 9. 何が公平で、何がまだ公平でないか

| 条件 | このNotebookの扱い |
| --- | --- |
| 初期データ | すべての手法で同一シード・同一介入集合・同一実測値 |
| 追加評価回数 | 同じ回数 |
| 真の目的値の算出 | 同じ既知SCMから`do`実行結果を評価 |
| 実験コスト | 介入集合の固定費用を全手法に適用、ただし**累積費用は一致させない** |
| 観測専用データ | GP-SCMにも追加していない |
| 利用可能な測定項目 | GP-SCMは`X,Z,M,Y`、集合別GPは介入値と`Y`など、モデルにより異なる |
| 最適化手法 | 同じ`sobol`設定。ただしMCサンプル数はGP-SCM固有 |
| 不確実性 | 介入集合別GPのGaussian posteriorと、GP-SCMのMC伝播は異なる |
| モデル仮定 | 既知DAG・機構不変性などの仮定を検証したものではない |

真の最適値は0と分かっているため、`simple_regret`が計算できます。
実際の製造問題ではこの最適値が分からないことが多く、その場合は
同じ定義のSimple Regretをそのまま算出できません。
また、2シードの平均と標準偏差は**計算例**であり、統計的な優位性の根拠にはなりません。

## 10. 手動の再現実験と次の検証課題

本格的な4方式比較はNotebookではなく既存ランナーを使います。

```bash
python benchmarks/causal_bo.py --seeds 0,1,2,3,4 \
    --n-initial-per-set 2 --iterations 5 \
    --include-gp-scm --mc-samples 128 --mc-seed 17
```

`--include-gp-scm`なしでは従来の3方式比較がそのまま実行されます。
出力には全評価の真値・simple regret・累積コストと、シード別集計が含まれます。
**例示した実行コマンドはベンチマークの再現手順であり、性能比較の実測結果を報告するものではありません。**

実務上は追加で（1）複数のDAGと非線形機構、（2）ノイズ・欠測・モデルミススペック、
（3）同じ費用上限での停止、（4）計算コストと精度、（5）GP-SCMにだけ
観測データを追加した場合を別条件として明示した比較を検討してください。
ただし、これらの実験をすべて実施したと主張するものではありません。